In [1]:
import torch
x = torch.randn(1000, 1000, device="cuda")
y = x @ x
print("Success:", y.shape)

Success: torch.Size([1000, 1000])


In [3]:
%pip install pandas transformers datasets torch scikit-learn seqeval gcsfs google-cloud-storage mlflow optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of fsspec[http] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of fsspec[http] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 101.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 91.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.7 MB/s eta 0:00:00
   ━━

In [ ]:


# pipeline.yaml structure to follow  
config = {
    "dataset": {
        "text_column": "review",
        "target_column": "adr_label"
    },
    "training": {
        "model_name": "emilyalsentzer/Bio_ClinicalBERT",
        "epochs": 3,
        "batch_size": 16
    },
    "paths": {
        # Input CSV files
    "raw_data": {
        "train": "../data/processed/train.csv",
        "val": "../data/processed/val.csv",
        "test": "../data/processed/test.csv"
    },
        # Folders for outputs 
    "models": {

    "../model/final_biobert_model"
    "models": "./biobert_adr_model"
    }
  ,
  "enriched_data": {
    "train": "../enriched_data/train_enriched",
    "val": "../enriched_data/val_enriched",
    "test": "../enriched_data/test_enriched"
  },
    "mlflow": {
        "experiment_name": "adr-nlp"
    }

}


In [ ]:
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    pipeline
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight


In [ ]:
import pandas as pd

# Load paths from yaml
train_df = pd.read_csv(config['paths']['raw_data']['train'])
val_df   = pd.read_csv(config['paths']['raw_data']['val'])
test_df  = pd.read_csv(config['paths']['raw_data']['test'])

train_df.head()

,uniqueID,drugName,condition,review,rating,date,usefulCount
0,40496,Savella,ibromyalgia,i039m on day 6 and i already feel the differen...,8,11-Jul-09,83
1,208687,Belsomra,Insomnia,didn039t work for me,1,28-Oct-17,2
2,228186,Etonogestrel,Birth Control,in my xprinc th only good thing about implanon...,2,25-Oct-09,3
3,17163,Flexeril,Muscle Spasm,flxril shorttrm mmory loss and xtrmly vivid dr...,5,6-Feb-11,47
4,26791,Ulipristal,Emergency Contraception,i took ella one 42 hours after having unprotec...,4,17-Jul-17,5


In [7]:
# Convert to HF Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# CLEAN DATA: Remove null reviews
train_dataset = train_dataset.filter(lambda x: x["review"] is not None and len(str(x["review"]).strip()) > 0)
val_dataset = val_dataset.filter(lambda x: x["review"] is not None and len(str(x["review"]).strip()) > 0)
test_dataset = test_dataset.filter(lambda x: x["review"] is not None and len(str(x["review"]).strip()) > 0)

print(f"Train samples after cleaning: {len(train_dataset)}")
print(f"Val samples after cleaning: {len(val_dataset)}")
print(f"Test samples after cleaning: {len(test_dataset)}")

Filter:   0%|          | 0/129037 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16130 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16130 [00:00<?, ? examples/s]

Train samples after cleaning: 129035
Val samples after cleaning: 16130
Test samples after cleaning: 16130


In [8]:
print(len(train_df), "→", len(train_dataset))
print(len(val_df), "→", len(val_dataset))
print(len(test_df), "→", len(test_dataset))

129037 → 129035
16130 → 16130
16130 → 16130


In [ ]:
from transformers import pipeline
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())

ner_pipeline = pipeline(
    "ner",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="simple",
    device=0   # use T4 GPU
)

def enrich_and_label_batched(examples):
    # internal pipeline batching on GPU
    batch_entities = ner_pipeline(
        examples["review"],
        batch_size=64
    )

    new_reviews = []
    adr_labels = []

    for text, entities in zip(examples["review"], batch_entities):
        entity_text = " ".join(ent["word"] for ent in entities)
        has_adr = any(ent["entity_group"] == "Sign_symptom" for ent in entities)

        new_reviews.append(f"{text} [ENT] {entity_text}")
        adr_labels.append(int(has_adr))

    examples["review"] = new_reviews
    examples["adr_label"] = adr_labels
    return examples


print("Processing Training Data...")
train_dataset = train_dataset.map(
    enrich_and_label_batched,
    batched=True,
    batch_size=128,           # map batch
    load_from_cache_file=False,
    desc="Train enrichment"
)

print("Processing Validation Data...")
val_dataset = val_dataset.map(
    enrich_and_label_batched,
    batched=True,
    batch_size=128,
    load_from_cache_file=False,
    desc="Validation enrichment"
)

print("Processing Test Data...")
test_dataset = test_dataset.map(
    enrich_and_label_batched,
    batched=True,
    batch_size=128,
    load_from_cache_file=False,
    desc="Test enrichment"
)

print("All datasets enriched")

train_dataset.save_to_disk(config['enriched_data']['train'])
val_dataset.save_to_disk(config['enriched_data']['val'])
test_dataset.save_to_disk(config['enriched_data']['test'])

GPU: Tesla T4
CUDA available: True


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

...

Test enrichment:   0%|          | 0/16130 [00:00<?, ? examples/s]

All datasets enriched


Saving the dataset (0/1 shards):   0%|          | 0/129035 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/16130 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/16130 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_from_disk

train_dataset = load_from_disk(config['enriched_data']['train'])
val_dataset = load_from_disk(config['enriched_data']['val'])
test_dataset = load_from_disk(config['enriched_data']['test'])

print(len(train_dataset), len(val_dataset), len(test_dataset))

129035 16130 16130


In [ ]:
from transformers import AutoTokenizer

# 1. Load the tokenizer matching your model
model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Define the tokenization function
def tokenize_func(examples):
    return tokenizer(
        examples["review"], 
        truncation=True, 
        padding="max_length", 
        max_length=256
    )

# 3. Apply tokenization to all 3 datasets
print("Tokenizing Train dataset...")
train_dataset = train_dataset.map(tokenize_func, batched=True)

print("Tokenizing Validation dataset...")
val_dataset = val_dataset.map(tokenize_func, batched=True)

print("Tokenizing Test dataset...")
test_dataset = test_dataset.map(tokenize_func, batched=True)

# 4. Target column mapping & formatting
# Hugging Face Trainer expects the label column to be specifically named 'labels'
train_dataset = train_dataset.rename_column("adr_label", "labels")
val_dataset = val_dataset.rename_column("adr_label", "labels")
test_dataset = test_dataset.rename_column("adr_label", "labels")

# PyTorch to strictly look at the numeric tensors it needs
columns_to_keep = ["input_ids", "attention_mask", "labels"]
train_dataset.set_format(type="torch", columns=columns_to_keep)
val_dataset.set_format(type="torch", columns=columns_to_keep)
test_dataset.set_format(type="torch", columns=columns_to_keep)

print("All datasets tokenized and ready for PyTorch!")


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizing Train dataset...


Map:   0%|          | 0/129035 [00:00<?, ? examples/s]

Tokenizing Validation dataset...


Map:   0%|          | 0/16130 [00:00<?, ? examples/s]

Tokenizing Test dataset...


Map:   0%|          | 0/16130 [00:00<?, ? examples/s]

All datasets tokenized and ready for PyTorch!


In [ ]:

import numpy as np
import torch
from torch import nn
from transformers import Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

# 1. Dynamically calculate class weights from the training set
print(" Calculating class weights for target balancing...")
labels = np.array(train_dataset["labels"])

weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
weights_tensor = torch.tensor(weights, dtype=torch.float)

# 2. Custom loss function wrapper for class weights
class CustomTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # works with DataParallel / multi-GPU
        device = next(model.parameters()).device

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(device)
        )

        loss = loss_fct(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

# 3. Function to compute classification metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

print("Custom Trainer and Evaluation framework initialized.")



pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.341175,0.314902,0.833333,0.898126,0.987455,0.823617
2,0.258880,0.333555,0.915667,0.951745,0.971952,0.932362


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

{'learning_rate': 2.3591264007393412e-05, 'num_train_epochs': 2, 'per_device_train_batch_size': 8}


In [ ]:

from transformers import AutoModelForSequenceClassification, TrainingArguments
import optuna

def model_init(trial):
    return AutoModelForSequenceClassification.from_pretrained(
        "emilyalsentzer/Bio_ClinicalBERT", 
        num_labels=2
    )

def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 4),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16]),
    }


args = TrainingArguments(
    output_dir="./hyperparam_search",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    dataloader_num_workers=2,
    logging_steps=100
)

small_train = train_dataset.shuffle(seed=42).select(range(12000))
small_val = val_dataset.shuffle(seed=42).select(range(3000))

trainer = CustomTrainer(
    class_weights=weights_tensor,
    model_init=model_init,
    args=args,
    train_dataset=small_train,
    eval_dataset=small_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

best_run = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=hp_space,
    n_trials=2
)

print(best_run.hyperparameters)


In [14]:
import mlflow
from transformers import TrainingArguments

final_args = TrainingArguments(
    output_dir="./final_biobert_model",
    num_train_epochs=best_run.hyperparameters["num_train_epochs"],
    per_device_train_batch_size=best_run.hyperparameters.get(
        "per_device_train_batch_size", 16
    ),
    learning_rate=best_run.hyperparameters["learning_rate"],
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    dataloader_num_workers=2
)

final_trainer = CustomTrainer(
    class_weights=weights_tensor,
    model=model_init(None),
    args=final_args,
    train_dataset=train_dataset,   # full training set
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

mlflow.set_experiment("adr-colab-experiment")

with mlflow.start_run():
    mlflow.log_params(best_run.hyperparameters)

    print("Training final model...")
    final_trainer.train()

    print("Running final evaluation on test set...")
    test_results = final_trainer.predict(test_dataset)

    mlflow.log_metrics(test_results.metrics)

    print("\nFinal Test Metrics:")
    print(test_results.metrics)

    print("Saving model...")
    final_trainer.save_model("./final_biobert_model")
    tokenizer.save_pretrained("./final_biobert_model")

print("Done.")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

Training final model...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.362450,0.309116,0.938004,0.964996,0.973721,0.956425
2,0.308177,0.410860,0.951767,0.973182,0.966982,0.979462


...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Running final evaluation on test set...



Final Test Metrics:
{'test_loss': 0.42216384410858154, 'test_accuracy': 0.9522628642281463, 'test_f1': 0.9736409694646035, 'test_precision': 0.9674149659863945, 'test_recall': 0.9799476295479603, 'test_runtime': 163.9607, 'test_samples_per_second': 98.377, 'test_steps_per_second': 6.154}
Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done.


In [15]:

# SAVE MODEL
trainer.save_model("./final_adr_model")
tokenizer.save_pretrained("./final_adr_model")
trainer.save_model("./best_f1_model")
tokenizer.save_pretrained("./best_f1_model")

AttributeError: 'NoneType' object has no attribute 'state_dict'

In [ ]:
import os
# Create a fresh directory
os.makedirs("./manual_final_save", exist_ok=True)

# Attempt to save the model and tokenizer from the trainer object
try:
    final_trainer.save_model("./manual_final_save")
    tokenizer.save_pretrained("./manual_final_save")
    print(" SUCCESS! The model is now on the Kaggle disk.")
except NameError:
    print(" Error: 'final_trainer' not found. Did the kernel restart?")
except Exception as e:
    print(f" Failed to save: {e}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ SUCCESS! The model is now on the Kaggle disk.


In [1]:
# Cloud Storage
from google.cloud import storage
storage_client = storage.Client(project='project-edc577ab-2a6a-4016-89e')
